# Praxis Colab Starter

Use this notebook to prepare the project on Colab compute and run your training entrypoint.

- If you are using VS Code with a mounted Colab server, leave `REPO_URL` blank.
- If you are using standalone Colab, set `REPO_URL` to your GitHub repo URL before running the setup cell.

In [ ]:
REPO_URL = "https://github.com/garypagangit/praxis.git"
BRANCH = "main"
WORKSPACE_DIR = "/content/praxis-workspace"


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

root = Path.cwd().resolve()

def run(command, cwd=None):
    print("$", " ".join(command))
    subprocess.run(command, cwd=str(cwd) if cwd else None, check=True)

def prepare_workspace(repo_url, branch, workspace_dir):
    if (root / "pyproject.toml").exists() and (root / "src").exists():
        target = root
    elif repo_url:
        target = Path(workspace_dir)
        if target.exists() and (target / ".git").exists():
            run(["git", "fetch", "origin", branch], cwd=target)
            run(["git", "checkout", branch], cwd=target)
            run(["git", "pull", "--ff-only", "origin", branch], cwd=target)
        else:
            if target.exists() and any(target.iterdir()):
                raise RuntimeError(
                    f"Workspace {target} exists and is not empty. Choose another WORKSPACE_DIR."
                )
            run(["git", "clone", "--branch", branch, repo_url, str(target)])
    else:
        raise RuntimeError(
            "Set REPO_URL for standalone Colab, or open this notebook from the repo workspace in VS Code."
        )

    requirements = target / "requirements.txt"
    if requirements.exists():
        run([sys.executable, "-m", "pip", "install", "-r", str(requirements)])
    run([sys.executable, "-m", "pip", "install", "-e", str(target)])

    src_dir = target / "src"
    if src_dir.exists() and str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

    os.chdir(target)
    return target

workspace = prepare_workspace(REPO_URL.strip() or None, BRANCH, WORKSPACE_DIR)
workspace


## Optional: mount Google Drive for checkpoints

Run this only inside Colab if you want outputs to persist in Drive.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    print("Drive mount is only available inside Colab.")
else:
    drive.mount("/content/drive")
    artifacts_dir = Path("/content/drive/MyDrive/praxis-runs")
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    print(f"Artifacts directory: {artifacts_dir}")


## Launch the starter run

This calls the packaged training entrypoint. Replace `src/praxis/train.py` with your real training loop when ready.

In [ ]:
!python -m praxis.train --config configs/example.json --run-name first-colab-run
